In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys, os
import matplotlib.pyplot as plt

In [ ]:
sys.path.append(os.path.abspath("../.."))
sys.path

In [ ]:
from src import utils

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('../../data/raw/emails.csv')

In [ ]:
df.head()

In [ ]:
df.shape

### convert column names to snakecase

In [ ]:
df = utils.convert_columns_to_snake_case(df)

## Detecting duplicate columns

In [ ]:
duplicates = []
cols = df.columns.tolist()

for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        c1, c2 = cols[i], cols[j]
        if df[c1].astype(str).equals(df[c2].astype(str)):
            duplicates.append((c1, c2))

for c1, c2 in duplicates:
    print(f"{c1}  ==  {c2}")

### Identifying numerical and categorical data

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df['time_to_renewal'].value_counts()

In [ ]:
df['time_to_renewal'].isnull().sum()

### in column 'time_to_renewal' we can find pre_renewal category and filter out data which is relevant to our use case

In [ ]:
df = df[df['time_to_renewal'] == 'pre_renewal']

In [ ]:
df.head()

In [ ]:
df.shape

### check for duplicates

In [ ]:
df.duplicated().sum()

### Most of the columns contain categorical data and mixed data such as numerical and string. 'Not Discussed' is used in place of nulls

In [ ]:
for col in df.columns:
    print(df[col].value_counts())
    print('nulls: ', df[col].isnull().sum())
    print('-' * 100)


In [ ]:
df.dtypes

## handle numerical columns

In [ ]:
numerical_cols = ['crm_contractor_sentiment_score', 'crm_agent_chase_count', 'crm_auto_renewal_status']

In [ ]:
df[numerical_cols].dtypes

In [ ]:
for col in numerical_cols:
    df[col] = pd.to_numeric(
        df[col], errors='coerce'
    )

In [ ]:
for col in numerical_cols:
    print(col)
    print(df[col].unique())
    print('-' * 100)

### cleaning crm_auto_renewal_status (a categorical column, already encoded, filling nulls)

In [ ]:
print(df['crm_auto_renewal_status'].isnull().sum())

In [ ]:
df['crm_auto_renewal_status'].value_counts()

In [ ]:
# fill with mode (most frequent)
df['crm_auto_renewal_status'] = df['crm_auto_renewal_status'].fillna(0)

In [ ]:
df['crm_auto_renewal_status'].value_counts()

### cleaning crm_contractor_sentiment_score

In [ ]:
df['crm_contractor_sentiment_score'].value_counts()

In [ ]:
df['crm_contractor_sentiment_score'].isnull().sum()

In [ ]:
print(df["crm_contractor_sentiment_score"].isna().mean() * 100, '%')

In [ ]:
print(df['crm_contractor_sentiment_score'].skew())

In [ ]:
plt.hist(df['crm_contractor_sentiment_score'], bins=10)
plt.xlabel('CRM Contractor Sentiment Score')
plt.ylabel('Frequency')
plt.title('Distribution of CRM Contractor Sentiment Scores')
plt.show()

### we cannot fill this directly with mean, median or mode as all of them will be 50 due to highest
This completely destroys the discriminative power of this feature.

In [ ]:
df['crm_contractor_sentiment'].value_counts()

In [ ]:
def clean_sentiment(val):
    if pd.isna(val):
        return "Unknown"
    
    val = str(val).lower().strip()
    
    # handle not discussed
    if "not" in val:
        return "Unknown"
    
    # handle mixed sentiment (very important)
    if "dissatisfied" in val and "neutral" in val:
        return "neutral"   # final state matters more
    
    # normal cases
    if "dissatisfied" in val:
        return "dissatisfied"
    elif "satisfied" in val:
        return "satisfied"
    
    
    return "neutral"

df['crm_contractor_sentiment'] = df['crm_contractor_sentiment'].apply(clean_sentiment)

In [ ]:
df['crm_contractor_sentiment'].value_counts()

we will add a missing indicator for null values and fill with -1 (outside 0-100 range), this helps model differentiate these values without compromizing on raws

In [ ]:
# missing indicator
df["sentiment_score_missing"] = df["crm_contractor_sentiment_score"].isna().astype(int)

# Fill "Not Discussed" group with a DISTINCT sentinel value
# Use a value OUTSIDE the normal 0-100 range, like -1
# This tells the model "this is a different category"
df["crm_contractor_sentiment_score"] = df["crm_contractor_sentiment_score"].fillna(-1)


In [ ]:
df.shape

In [ ]:
print(df['crm_contractor_sentiment_score'].isnull().sum())

In [ ]:
df['crm_contractor_sentiment_score'].value_counts()

## crm_agent_chase_count

In [ ]:
print(df['crm_agent_chase_count'].unique())

In [ ]:
df['crm_agent_chase_count'].value_counts()

In [ ]:
print(df['crm_agent_chase_count'].isnull().sum())

In [ ]:
plt.hist(df['crm_agent_chase_count'], bins=20)
plt.xlabel('crm_agent_chase_count')
plt.ylabel('frequency')
plt.show()

In [ ]:
df['crm_agent_chase_count'].skew()

In [ ]:
df['crm_agent_chase_count'].median()

In [ ]:
# use median to fill
df['crm_agent_chase_count'] = df['crm_agent_chase_count'].fillna(
    df['crm_agent_chase_count'].median()
)

In [ ]:
plt.hist(df['crm_agent_chase_count'], bins=20)
plt.xlabel('crm_agent_chase_count')
plt.ylabel('frequency')
plt.show()

### handling categorical features

In [ ]:
for col in df.columns:
    print(df[col].value_counts())
    print(df[col].isnull().sum())

In [ ]:
categorical_cols = []

for col in df.columns:
    if col not in numerical_cols:
        categorical_cols.append(col)

## GROUP A: Clean columns — just fill NaN with "Unknown"

In [ ]:
group_a_cols = [
    'crm_accreditation_completed',
    'crm_timely_completion',
    'crm_progress_towards_accreditation',
    'crm_contractor_suggested_leave',
    'crm_customer_payment_intention',
    'crm_competitors_mentioned',
    'crm_platform_issues_raised',
    'crm_agent_chased_contractor',
    'crm_accreditation_issues',
]

In [ ]:
for col in group_a_cols:
    print(df[col].value_counts())
    print("nulls: ", df[col].isnull().sum())
    print('-' * 100)

In [ ]:
def clean_ternary_column(val, garbage_fallback="Unknown"):
    """
    Cleans a Yes/No/Not Discussed categorical column.

    Rules (in order):
      1. NaN → "Unknown"
      2. "Yes" / "No" / "Not Discussed" (exact match after strip) → kept as-is
      3. "Not applicable..." / "Not Applicable..." → "Not Discussed"
      4. "[Yes/No]" / "[Yes/No/Not Discussed]" → "Unknown"
      5. Anything else (garbage text, numbers) → garbage_fallback
    """
    if pd.isna(val):
        return "Unknown"

    val_clean = str(val).strip()

    # Rule 2: exact match for valid categories
    if val_clean in ("Yes", "No", "Not Discussed"):
        return val_clean

    # Rule 3: "Not applicable" variations → "Not Discussed"
    val_lower = val_clean.lower()
    if val_lower.startswith("not applicable"):
        return "Not Discussed"

    # Rule 4: ambiguous bracketed placeholders → "Unknown"
    if val_clean.startswith("["):
        return "Unknown"

    # Rule 5: everything else → column-specific fallback
    return garbage_fallback


In [ ]:
for col in group_a_cols:
    df[col] = df[col].apply(clean_ternary_column)

In [ ]:
for col in group_a_cols:
    print(df[col].value_counts())
    print("nulls: ", df[col].isnull().sum())
    print('-' * 100)

### GROUP B: Columns with garbage text to clean

In [ ]:
group_b_cols = [
    'crm_delays_in_accreditation',
    'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned',
    'crm_membership_overdue',
    'crm_dissatisified_with_renewal_price',
    'crm_customer_complained',
    'crm_refund_mentioned',
    'crm_negative_customer_experience',
    'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned'
]

In [ ]:
for col in group_b_cols:
    print(df[col].value_counts())
    print(df[col].isnull().sum())
    print('_'*100)

In [ ]:
group_b_config = {
    'crm_delays_in_accreditation':            'Yes',
    'crm_contractor_engagement':              'No',
    'crm_dts_or_ssip_mentioned':              'No',
    'crm_membership_overdue':                 'Yes',
    'crm_dissatisified_with_renewal_price':   'No',
    'crm_customer_complained':                'No',
    'crm_refund_mentioned':                   'No',
    'crm_negative_customer_experience':       'Yes',
    'crm_dissatisfaction_with_support':       'Yes',
    'crm_financial_hardship_mentioned':       'Yes',
}

for col, fallback in group_b_config.items():
    df[col] = df[col].apply(lambda v, fb=fallback: clean_ternary_column(v, garbage_fallback=fb))


In [ ]:
for col in group_b_cols:
    print(df[col].value_counts())
    print(df[col].isnull().sum())
    print('_'*100)

### GROUP C: crm_membership_level — consolidate categories

In [ ]:
df['crm_membership_level'].value_counts()

In [ ]:
df['crm_membership_level'].isnull().sum()

In [ ]:
def clean_membership_level(val):
    """
    Consolidate crm_membership_level into clean categories.
    NaN → "Unknown"
    """
    if pd.isna(val):
        return "Unknown"

    val_clean = str(val).strip()

    # Exact matches for main categories
    if val_clean in ("Not Discussed", "Accredited", "In progress", "Members only", "Not Accredited"):
        return val_clean

    # Consolidate rare variants
    val_lower = val_clean.lower()
    if 'standard' in val_lower or 'express' in val_lower or val_lower == 'membership (standard)':
        return "Standard"
    if 'premier' in val_lower or val_lower == 'prem':
        return "Premier"
    if val_lower in ('gold', 'silver', 'accredited (gold level)'):
        return "Accredited"

    return "Unknown"

df['crm_membership_level'] = df['crm_membership_level'].apply(clean_membership_level)

In [ ]:
df['crm_membership_level'].value_counts()

### VERIFICATION: Check all categorical columns are clean

In [ ]:
categorical_cols = [
    'crm_accreditation_completed', 'crm_timely_completion',
    'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_contractor_sentiment', 'crm_dts_or_ssip_mentioned',
    'crm_customer_payment_intention', 'crm_competitors_mentioned',
    'crm_membership_level', 'crm_platform_issues_raised',
    'crm_agent_chased_contractor', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_dissatisified_with_renewal_price',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]

In [ ]:
print("=== POST-CLEANING VERIFICATION ===\n")
for col in categorical_cols:
    print(f"{col}")
    print(f"  unique: {df[col].nunique()}")
    print(f"  nulls:  {df[col].isnull().sum()}")
    print(f"  values: {df[col].value_counts().to_dict()}")
    print()
print(f"Final shape: {df.shape}")
print(f"Rows removed: 0")

In [ ]:
df.isnull().sum()

### save df as a csv file in data/intrim/

In [ ]:
df.to_csv("../../data/interim/cleaned_emails.csv", index=False)